# Workflow 19: sensitivity to pHash curation
ResNet50 with ImageNet weights, the adapted protocol, and training seeds 42, 43 and 44. Six new training runs take approximately 3–5 hours. Three historical runs can be reused after checking image identities, split roles, training settings and source hashes. Select **BRISC Linux (GPU)** and run the cells in order. Do not run another GPU experiment at the same time.

The conditions are `restored_guarded` (restore excluded training images while preventing identical RGB pixels across partitions), `exact_only` (also remove identical RGB copies within training), and `historical_guarded` (retain the historically included training images with the same safeguards).

Validation is fixed from workflow 17. The common test uses historically retained test images. Validation and test are deduplicated by decoded RGB pixels, giving test priority; groups with conflicting labels are excluded and counted. Historically excluded test images are not restored. This evaluates sensitivity to training-set composition and size on a common test, not the full effect of curation or patient independence.

Hyperparameters are fixed from workflow 17. Report every seed without choosing the best test result. Existing training results are reused only if the compatibility checks pass; `reuse.json` records their provenance. The original manuscript results remain separate.

In [ ]:
from pathlib import Path
import sys, subprocess, os, json
from datetime import datetime
import pandas as pd
ROOT=Path('/mnt/c/Users/cbot/Desktop/brain_tumor_brisc_tmp')
SCRIPT=ROOT/'scripts/phash_sensitivity.py'
assert sys.platform == 'linux', 'Select BRISC Linux (GPU)'
subprocess.run([sys.executable,'-c',"import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g"],check=True)


In [ ]:
# To resume after restarting the kernel, set OUT to the previous output directory.
if 'OUT' not in globals():
    OUT=ROOT/'runs'/('phash_sensitivity_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
if not (OUT/'design.json').exists():
    subprocess.run([sys.executable,str(SCRIPT),'prepare','--output',str(OUT)],check=True,cwd=ROOT)
print('Results:', OUT)
display(pd.read_csv(OUT/'counts.csv').pivot_table(index='arm',columns='role',values='n',aggfunc='sum'))


## Training
Run the next cell and allow it to finish. Epoch progress is displayed and a log is saved for each job. Interrupting the cell terminates its child process. Completed jobs are skipped when resuming. Incomplete output directories stop execution for inspection; results are not deleted.

In [ ]:
design=json.loads((OUT/'design.json').read_text())
for seed in design['seeds']:
    for arm in design['arms']:
        job=f'imagenet_adapted_seed{seed}'
        target=OUT/arm/job
        if (target/'metrics.json').exists():
            print('Completed; skip:',arm,job); continue
        if target.exists():
            raise RuntimeError(f'Incomplete run: {target}. Inspect before resuming.')
        print('START',arm,job,flush=True)
        env=dict(os.environ,PYTHONHASHSEED=str(seed),TF_DETERMINISTIC_OPS='1')
        with (OUT/f'{arm}_{job}.log').open('w') as log:
            process=subprocess.Popen([sys.executable,'-u',str(SCRIPT),'run','--output',str(OUT),'--arm',arm,'--job',job],cwd=ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            try:
                for line in process.stdout:
                    print(line,end='',flush=True); log.write(line); log.flush()
                if process.wait()!=0: raise RuntimeError(f'Failed: {arm} {job}. See the log.')
            except BaseException:
                if process.poll() is None:
                    process.terminate()
                    try: process.wait(timeout=10)
                    except subprocess.TimeoutExpired: process.kill(); process.wait()
                raise


In [ ]:
subprocess.run([sys.executable,str(SCRIPT),'summarize','--output',str(OUT)],check=True,cwd=ROOT)
display(pd.read_csv(OUT/'summary.csv'))
display(pd.read_csv(OUT/'paired_differences.csv'))
print('Report:',OUT/'REPORT_FOR_COAUTHORS.md')
